# Lab 1.5 — End-to-End AI Requirements Classification Agent (Gemini API)

**Thesis:** *Application of Machine Learning Methods for Optimization of Requirements Specification Processes*

**Goal:** Demonstrate a **single AI agent** that accepts a new natural-language software requirement **X**, performs **data understanding → inference → reasoning → output generation**, and returns **y = FR or NFR**, plus a concise explanation.

Two prompting strategies are tested on the **same unseen input**: **zero-shot** and **few-shot** (using five known labelled pairs from Lab 1.2 / `data/examples.csv`). The results are saved in CSV, JSON, and Markdown for GitHub submission.

> **Scope:** This is an **LLM-based demonstration using the Gemini API**. Your thesis's TF-IDF/SVM/Random Forest classifiers are *not run* in this notebook. The LLM performs the inference and explanation; do not report these results as SVM/RF predictions. An actual model comparison would require loading and invoking the trained models.

**Before running:** Obtain a Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey). Do **not** commit or share your key. In Google Colab's left **Secrets (key)** panel, create a secret named `GEMINI_API_KEY` and allow notebook access. Alternatively, the notebook can prompt for the key securely.

## 1. Install and import dependencies
This uses Google's supported `google-genai` Python SDK. API access requires a valid key, access to a compatible model, and available quota.

In [ ]:
%pip -q install -U google-genai pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.4.0 which is incompatible.


In [ ]:
import os, json, re, csv, getpass
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from google import genai
from google.genai import types
from IPython.display import display, Markdown

OUTPUT_DIR = Path('lab_1_5_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Ready. Output folder:', OUTPUT_DIR.resolve())

Ready. Output folder: /content/lab_1_5_outputs


## 2. Set up Gemini API securely
If you get **404 MODEL_NOT_FOUND**, choose a model that is available for your account. If you get **429 RESOURCE_EXHAUSTED**, check your API quota. The diagnostic cell shows the actual error instead of hiding it.

In [ ]:

from google.colab import userdata

GEMINI_KEY = userdata.get("GEMINI_KEY")

print("SUCCESS: API key loaded securely!")




SUCCESS: API key loaded securely!


In [ ]:

from google.colab import userdata
from google import genai

# Retrieve the key securely
GEMINI_KEY = userdata.get("GEMINI_KEY")

client = genai.Client(api_key=GEMINI_KEY.strip())
MODEL = "gemini-3.5-flash-lite"

try:
    response = client.models.generate_content(
        model=MODEL,
        contents="Reply only with OK."
    )
    print("SUCCESS: Gemini API connected!")
    print("Response:", response.text)

except Exception as e:
    print("Error:", str(e)[:500])




SUCCESS: Gemini API connected!
Response: OK.


In [ ]:

# Use the existing Gemini client
# initialized in your earlier cell.

preferred_models = [
    "gemini-3.8-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite"
]

MODEL = None

for candidate in preferred_models:
    try:
        response = client.models.generate_content(
            model=candidate,
            contents="Reply only with OK."
        )

        print("SUCCESS!")
        print("Working model:", candidate)
        print("Gemini response:", response.text)

        MODEL = candidate
        break

    except Exception as e:
        print(candidate, "failed:")
        print(str(e)[:300])

if MODEL is None:
    print("No preferred model passed the test.")
    print("Check the model names listed by your account.")


gemini-3.8-flash failed:
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
SUCCESS!
Working model: gemini-3.5-flash-lite
Gemini response: OK


In [ ]:

# Reuse the working Gemini client and MODEL

X = (
    "The application shall maintain 99.9% "
    "availability during each calendar month."
)

zero_shot = f"""
You are an AI Requirements Classification Agent.

Classify the following software requirement
as FR (Functional Requirement) or
NFR (Non-Functional Requirement).

Input X: {X}

Return exactly:
y: FR or NFR
explanation: one sentence
"""

few_shot = f"""
You are an AI Requirements Classification Agent.

Learn from these labelled examples:

X: Allow users to create a new account
y: FR

X: Generate monthly financial reports
y: FR

X: Respond to user requests within 2 seconds
y: NFR

X: Encrypt stored customer information
y: NFR

X: Send email notifications after registration
y: FR

Now classify this new requirement:

X: {X}

Return exactly:
y: FR or NFR
explanation: one sentence
"""

for name, prompt in [
    ("Zero-shot", zero_shot),
    ("Few-shot", few_shot)
]:
    print("\n", "=" * 35)
    print(name, "Result")
    print("=" * 35)

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )
        print(response.text)

    except Exception as e:
        print("Error:", str(e)[:500])



Zero-shot Result
y: NFR
explanation: This requirement specifies the system's availability and reliability performance standard rather than a specific business function.

Few-shot Result
y: NFR
explanation: This requirement specifies a system availability and reliability metric rather than a specific functional feature.


In [ ]:
try:
    probe = client.models.generate_content(model=MODEL, contents='Respond only with OK.')
    print('Gemini connection:', (probe.text or '').strip())
except Exception as exc:
    print('Gemini connection FAILED:', type(exc).__name__)
    print(str(exc)[:1300])
    print('Check API key (401/403), model ID (404), quota (429), or service availability (503).')
    raise

Gemini connection: OK


## 3. Input data: five known labelled (X, y) pairs
These are the five examples the user supplied from their earlier work. If your GitHub CSV has different wording, upload the original `data/examples.csv` and replace this data before final submission. **Never include the new evaluation input in the few-shot examples.**

In [ ]:

import pandas as pd
from IPython.display import display

# Five labelled examples from Lab 1.2
examples = [
    {
        "X": "The system shall allow users to create a new account.",
        "y": "FR"
    },
    {
        "X": "The application shall generate monthly financial reports.",
        "y": "FR"
    },
    {
        "X": "The software shall respond to user requests within 2 seconds.",
        "y": "NFR"
    },
    {
        "X": "The system shall encrypt stored customer information.",
        "y": "NFR"
    },
    {
        "X": "The application shall send email notifications.",
        "y": "FR"
    }
]

# Create DataFrame
examples_df = pd.DataFrame(examples)

# Display without relying on the display formatter
print("Five Known Labelled Examples:")
print(examples_df.to_string(index=False))

# New test requirement
X_NEW = (
    "The application shall maintain 99.9% "
    "availability during each calendar month."
)

EXPECTED_Y = "NFR"

# Verify that the new requirement is not in the examples
assert X_NEW not in examples_df["X"].tolist()

print("\nNew Input X:", X_NEW)
print("Expected Classification:", EXPECTED_Y)


Five Known Labelled Examples:
                                                            X   y
        The system shall allow users to create a new account.  FR
    The application shall generate monthly financial reports.  FR
The software shall respond to user requests within 2 seconds. NFR
        The system shall encrypt stored customer information. NFR
              The application shall send email notifications.  FR

New Input X: The application shall maintain 99.9% availability during each calendar month.
Expected Classification: NFR


### Optional: Use your GitHub CSV instead
Run this cell **only if** you want to upload the original CSV. It supports CSV column names `X,y`, `requirement,label`, or `text,classification`. Then re-run later cells.

In [ ]:
# OPTIONAL: Uncomment this cell content if you want to upload the original examples.csv.
# from google.colab import files
# uploaded = files.upload()
# csv_path = next((name for name in uploaded if name.lower().endswith('.csv')), None)
# if csv_path:
#     df = pd.read_csv(csv_path)
#     print('Columns:', list(df.columns))
#     candidates = [('X','y'), ('requirement','label'), ('text','classification')]
#     pair = next(((a,b) for a,b in candidates if a in df.columns and b in df.columns), None)
#     if pair is None:
#         raise ValueError('Use CSV columns X,y or requirement,label or text,classification')
#     examples = [dict(X=str(x).strip(), y=str(y).strip().upper()) for x,y in df[list(pair)].itertuples(index=False, name=None)]
#     assert all(e['y'] in {'FR','NFR'} for e in examples)
#     assert all(e['X'] != X_NEW for e in examples), 'Test input must not be a training example'
#     examples_df = pd.DataFrame(examples)
#     display(examples_df)

## 4. Define the single agent's workflow
The agent follows all four stages, but **one Gemini API call** handles interpretation, classification, reasoning, and formatting. The stage trace is a description of the agent's high-level process, **not hidden chain-of-thought**.

In [ ]:
AGENT_INSTRUCTIONS = '''You are a single AI Requirements Classification Agent.
Classify exactly one natural-language software requirement.
FR = a required software function, behavior, or service.
NFR = a quality attribute or constraint (performance, security, reliability, availability, etc.).
Follow these four stages internally:
1. Data understanding: recognize the input and its key observable clues.
2. Inference: predict FR or NFR.
3. Reasoning: provide one concise, observable justification (not private reasoning steps).
4. Output generation: return exactly one JSON object with keys "y" and "explanation".
Do not claim to have invoked TF-IDF, SVM, or Random Forest.
Use "y" strictly as FR or NFR. Keep explanation to one sentence.
'''

def make_prompt(x, shots=None):
    parts = [AGENT_INSTRUCTIONS]
    if shots:
        parts.append('Known labelled examples (X, y):')
        for item in shots:
            parts.append(f'X: {item["X"]}\ny: {item["y"]}')
        parts.append('Now classify a NEW input; do not merely repeat a known example.')
    parts.append(f'New input X:\n{x}')
    return '\n\n'.join(parts)

zero_prompt = make_prompt(X_NEW)
few_prompt = make_prompt(X_NEW, examples)
print('Zero-shot prompt length:', len(zero_prompt))
print('Few-shot prompt length:', len(few_prompt))

Zero-shot prompt length: 807
Few-shot prompt length: 1233


## 5. Run Prompt 1: zero-shot
Gemini receives only the classification instructions and the new input X — **no labelled examples**.

In [ ]:
def invoke_agent(prompt, method):
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0,
                response_mime_type='application/json',
                response_schema={
                    'type': 'OBJECT',
                    'properties': {
                        'y': {'type': 'STRING', 'enum': ['FR','NFR']},
                        'explanation': {'type': 'STRING'}
                    },
                    'required': ['y','explanation']
                }
            )
        )
        raw = response.text or ''
        parsed = json.loads(raw)
        if parsed.get('y') not in ('FR', 'NFR'):
            raise ValueError('Model returned an invalid classification: ' + str(parsed.get('y')))
        return {'method':method, 'X':X_NEW, 'y':parsed['y'],
                'explanation':parsed.get('explanation',''),
                'expected_y':EXPECTED_Y, 'matches_reference':parsed['y']==EXPECTED_Y,
                'model':MODEL, 'status':'success'}
    except Exception as exc:
        print(f'{method} failed: {type(exc).__name__}: {str(exc)[:1400]}')
        raise

# Run zero-shot classification
zero_result = invoke_agent(zero_prompt, "Zero-shot")

# Print results without the DataFrame display formatter
print("\n===== ZERO-SHOT RESULT =====")
print("Method:", zero_result["method"])
print("Input X:", zero_result["X"])
print("Predicted y:", zero_result["y"])
print("Explanation:", zero_result["explanation"])
print("Matches reference:",
      zero_result["matches_reference"])
print("Model:", zero_result["model"])
print("Status:", zero_result["status"])


===== ZERO-SHOT RESULT =====
Method: Zero-shot
Input X: The application shall maintain 99.9% availability during each calendar month.
Predicted y: NFR
Explanation: The requirement specifies a system availability percentage, which is a quality attribute related to reliability and uptime.
Matches reference: True
Model: gemini-3.5-flash-lite
Status: success


## 6. Run Prompt 2: few-shot
Gemini receives the **same unseen X**, along with the known labelled (X,y) examples before making its prediction.

In [ ]:

# Run the few-shot AI agent
few_result = invoke_agent(few_prompt, "Few-shot")

print("\n===== FEW-SHOT CLASSIFICATION =====")
print("Method:", few_result["method"])
print("Input X:", few_result["X"])
print("Predicted y:", few_result["y"])
print("Explanation:", few_result["explanation"])
print("Matches reference:",
      few_result["matches_reference"])



===== FEW-SHOT CLASSIFICATION =====
Method: Few-shot
Input X: The application shall maintain 99.9% availability during each calendar month.
Predicted y: NFR
Explanation: The requirement specifies a system availability percentage, which is a reliability quality attribute rather than a functional behavior.
Matches reference: True


## 7. Compare outcomes
For one observation, `matches_reference` only indicates agreement with a human-selected expected answer; **it is not classification accuracy** on the thesis dataset.

In [ ]:

# Task 1.5 - Compare Zero-Shot and Few-Shot Results

print("=" * 65)
print("TASK 1.5 - GEMINI AI CLASSIFICATION RESULTS")
print("=" * 65)

results = [zero_result, few_result]

for result in results:
    print("\nMethod:", result["method"])
    print("Input X:", result["X"])
    print("Predicted y:", result["y"])
    print("Expected y:", result["expected_y"])
    print("Explanation:", result["explanation"])
    print("Matches Reference:", result["matches_reference"])
    print("Model:", result["model"])
    print("Status:", result["status"])
    print("-" * 65)

# Verify both prompts used the same input
same_input = len(set(r["X"] for r in results)) == 1

print("\nBoth methods used identical X:", same_input)

# Count agreement with expected classification
correct = sum(r["matches_reference"] for r in results)

print("Predictions matching reference:",
      f"{correct}/{len(results)}")


TASK 1.5 - GEMINI AI CLASSIFICATION RESULTS

Method: Zero-shot
Input X: The application shall maintain 99.9% availability during each calendar month.
Predicted y: NFR
Expected y: NFR
Explanation: The requirement specifies a system availability percentage, which is a quality attribute related to reliability and uptime.
Matches Reference: True
Model: gemini-3.5-flash-lite
Status: success
-----------------------------------------------------------------

Method: Few-shot
Input X: The application shall maintain 99.9% availability during each calendar month.
Predicted y: NFR
Expected y: NFR
Explanation: The requirement specifies a system availability percentage, which is a reliability quality attribute rather than a functional behavior.
Matches Reference: True
Model: gemini-3.5-flash-lite
Status: success
-----------------------------------------------------------------

Both methods used identical X: True
Predictions matching reference: 2/2


## 8. Save assignment evidence and download
Use the outputs in your GitHub repo under `results/lab_1_5/`. Record the *real API response*, not an expected example. Keep your Gemini key out of the repository.

In [ ]:

import csv
import json
import shutil
from pathlib import Path
from google.colab import files

# Create output folder
OUTPUT_DIR = Path("lab_1_5_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Collect actual Gemini results
results = [zero_result, few_result]

# 1. Save CSV results
csv_file = OUTPUT_DIR / "gemini_results.csv"

fields = [
    "method", "X", "y", "explanation",
    "expected_y", "matches_reference",
    "model", "status"
]

with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for result in results:
        writer.writerow({
            key: result.get(key, "")
            for key in fields
        })

# 2. Save JSON results
with open(
    OUTPUT_DIR / "gemini_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# 3. Save Zero-Shot Prompt
with open(
    OUTPUT_DIR / "zero_shot_prompt.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(zero_prompt)

# 4. Save Few-Shot Prompt
with open(
    OUTPUT_DIR / "few_shot_prompt.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(few_prompt)

# 5. Generate report
report = f"""
# Task 1.5 - Gemini AI Agent Results

## Thesis
Application of Machine Learning Methods for
Optimization of Requirements Specification Processes

## AI Model
{MODEL}

## Input X
{X_NEW}

## Zero-Shot Result
Prediction: {zero_result['y']}
Explanation: {zero_result['explanation']}

## Few-Shot Result
Prediction: {few_result['y']}
Explanation: {few_result['explanation']}

## Conclusion
The Gemini API was successfully integrated into
Google Colab.

Both zero-shot and few-shot prompting methods
classified the input requirement as a
Non-Functional Requirement (NFR).

The demonstration evaluates Gemini prompting,
not the original SVM or Random Forest models.
"""

with open(
    OUTPUT_DIR / "report.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(report)

# 6. Create ZIP file
archive = shutil.make_archive(
    "lab_1_5_outputs",
    "zip",
    root_dir=OUTPUT_DIR
)

print("Results saved successfully!")
print("Generated files:")

for file in OUTPUT_DIR.iterdir():
    print("-", file.name)

# 7. Download ZIP
files.download(archive)


Results saved successfully!
Generated files:
- zero_shot_prompt.txt
- gemini_results.csv
- few_shot_prompt.txt
- gemini_results.json
- report.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Suggested assignment conclusion (edit after executing)

> This Google Colab demonstration implemented a single AI Requirements Classification Agent using the Gemini API. A natural-language software requirement was submitted as input X. Gemini processed the input using the specified data-understanding, inference, reasoning, and output-generation stages, and returned an FR/NFR label y with a one-sentence explanation. Zero-shot and few-shot prompts were applied to the same unseen requirement, with the few-shot prompt additionally containing five previously labelled examples. The observed results were exported to CSV, JSON, and Markdown for reproducibility. The LLM-based experiment is a conceptual extension of the thesis's classical ML pipeline and does not constitute execution of the trained SVM or Random Forest classifiers.

**GitHub submission:** Upload this `.ipynb` under `notebooks/` and its generated outputs under `results/lab_1_5/`.

**References:** [Google Gen AI Python SDK](https://googleapis.github.io/python-genai/) · [Google AI Studio](https://aistudio.google.com/apikey).

In [1]:

# Lab 1.5 - Integration of Lab 1.4 Prompts
# Reuse the existing Gemini client and MODEL.

X = (
    "The application shall maintain 99.9% "
    "availability during each calendar month."
)

examples = [
    ("The system shall allow users to create a new account.", "FR"),
    ("The application shall generate monthly financial reports.", "FR"),
    ("The software shall respond to user requests within 2 seconds.", "NFR"),
    ("The system shall encrypt stored customer information.", "NFR"),
    ("The application shall send email notifications.", "FR")
]

instructions = """
You are a single AI Requirements Classification Agent.

Process the input through:
1. Data understanding: identify relevant textual clues.
2. Reasoning: explain the relevant requirement characteristics.
3. Inference: classify the requirement as FR or NFR.
4. Output generation: return the final classification.

FR = Functional Requirement
NFR = Non-Functional Requirement

Return exactly:
y: FR or NFR
explanation: one concise sentence

Do not claim to have executed SVM or Random Forest.
"""

# Prompt 1 - Zero-shot
zero_prompt = f"""
{instructions}

New input X:
{X}
"""

# Prompt 2 - Few-shot
known_pairs = "\n\n".join(
    f"X: {requirement}\ny: {label}"
    for requirement, label in examples
)

few_prompt = f"""
{instructions}

Known labelled examples from Lab 1.4:
{known_pairs}

Now classify this new input X:
{X}
"""

print("Both Lab 1.4 prompts are ready.")
print("New input X:", X)
print("Few-shot examples:", len(examples))


Both Lab 1.4 prompts are ready.
New input X: The application shall maintain 99.9% availability during each calendar month.
Few-shot examples: 5


In [2]:

# Execute both prompts using the Gemini API

results = {}

for method, prompt in [
    ("Zero-shot", zero_prompt),
    ("Few-shot", few_prompt)
]:
    print("\n" + "=" * 55)
    print("METHOD:", method)
    print("=" * 55)

    print("\n[1] DATA UNDERSTANDING")
    print("Input X:", X)

    print("\n[2] REASONING")
    print("Gemini will identify requirement characteristics.")

    print("\n[3] INFERENCE")
    print("Requesting FR/NFR prediction from Gemini...")

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )

        output = response.text or ""

        print("\n[4] OUTPUT GENERATION")
        print(output)

        results[method] = output

    except Exception as e:
        print("Gemini API error:", str(e)[:500])

print("\nBoth prompt executions finished.")



METHOD: Zero-shot

[1] DATA UNDERSTANDING
Input X: The application shall maintain 99.9% availability during each calendar month.

[2] REASONING
Gemini will identify requirement characteristics.

[3] INFERENCE
Requesting FR/NFR prediction from Gemini...
Gemini API error: name 'client' is not defined

METHOD: Few-shot

[1] DATA UNDERSTANDING
Input X: The application shall maintain 99.9% availability during each calendar month.

[2] REASONING
Gemini will identify requirement characteristics.

[3] INFERENCE
Requesting FR/NFR prediction from Gemini...
Gemini API error: name 'client' is not defined

Both prompt executions finished.


### Integration of Lab 1.4 Prompt Engineering

The two prompts developed in Lab 1.4 were integrated into Google Colab and executed using the Gemini API.

The zero-shot prompt classified a new software requirement without any labelled examples. The few-shot prompt used five known Functional and Non-Functional Requirement examples before classifying the same new requirement.

The single AI agent follows four conceptual stages: data understanding, reasoning, inference, and output generation. The Gemini model analyzes the natural-language requirement, predicts its category, and generates an explanation.

In the demonstrated experiment, both prompting methods classified the 99.9% availability requirement as NFR. These results show that the Gemini-based agent can perform end-to-end FR/NFR classification for the selected test observation.

## End-to-End AI Requirements Classification Pipeline

This experiment demonstrates a single AI Requirements Classification Agent developed as a conceptual extension of the thesis "Application of Machine Learning Methods for Optimization of Requirements Specification Processes."

**Stage 1 — Data Understanding:** The agent receives a natural-language requirement and identifies relevant information, such as functional behavior or quality constraints.

**Stage 2 — Reasoning:** The Gemini model generates a concise explanation based on the identified characteristics of the requirement.

**Stage 3 — Inference:** The agent predicts whether the requirement is Functional (FR) or Non-Functional (NFR).

**Stage 4 — Output Generation:** The agent returns the predicted label and a natural-language explanation in a structured format.

**Input X:** A new natural-language software requirement.

**Output y:** FR or NFR, accompanied by an explanation.

The Gemini agent is a generative-AI demonstration of the classification task. The thesis's trained SVM and Random Forest models are not executed in this pipeline.

In [4]:

from google.colab import userdata
from google import genai

# Load Gemini API key securely
GEMINI_KEY = userdata.get("GEMINI_KEY")

# Initialize Gemini client
client = genai.Client(
    api_key=GEMINI_KEY.strip()
)

# Previously working Gemini model
MODEL = "gemini-3.5-flash-lite"

print("Gemini client initialized successfully!")
print("Selected model:", MODEL)


Gemini client initialized successfully!
Selected model: gemini-3.5-flash-lite


In [5]:

import json
import re

# Reuse the connected Gemini client and model.
# client = genai.Client(api_key=GEMINI_KEY)
# MODEL = "gemini-3.5-flash-lite"

# ------------------------------
# INPUT X
# ------------------------------
X = (
    "The application shall maintain 99.9% "
    "availability during each calendar month."
)

print("=" * 60)
print("END-TO-END AI REQUIREMENTS CLASSIFICATION")
print("=" * 60)

# ------------------------------
# STAGE 1 - DATA UNDERSTANDING
# ------------------------------
print("\n[STAGE 1] DATA UNDERSTANDING")
print("Original Input X:", X)

# Basic preprocessing for observable logging
cleaned_X = re.sub(r"\s+", " ", X).strip()
print("Normalized Input:", cleaned_X)

# ------------------------------
# STAGE 2 - REASONING
# ------------------------------
print("\n[STAGE 2] REASONING")
print("Requesting a brief explanation from Gemini...")

prompt = f"""
You are a single AI Requirements Classification Agent.

Analyze the following software requirement.

Input X:
{cleaned_X}

Perform these tasks:
1. Identify the relevant functional or quality clues.
2. Explain the classification in one concise sentence.
3. Predict FR or NFR.

FR = Functional Requirement.
NFR = Non-Functional Requirement.

Return ONLY valid JSON with these fields:
{{
  "clues": "brief observable clues",
  "y": "FR or NFR",
  "explanation": "one sentence"
}}

Do not claim to have executed SVM or Random Forest.
"""

# ------------------------------
# STAGE 3 - INFERENCE
# ------------------------------
print("\n[STAGE 3] INFERENCE")
print("Sending requirement to Gemini API...")

response = client.models.generate_content(
    model=MODEL,
    contents=prompt
)

raw_output = (response.text or "").strip()

# Handle optional Markdown code fences
raw_output = re.sub(
    r"^```(?:json)?\s*|\s*```$",
    "",
    raw_output
).strip()

result = json.loads(raw_output)

if result.get("y") not in ("FR", "NFR"):
    raise ValueError("Invalid classification returned.")

# ------------------------------
# STAGE 4 - OUTPUT GENERATION
# ------------------------------
print("\n[STAGE 4] OUTPUT GENERATION")
print("Identified clues:", result.get("clues"))
print("Predicted output y:", result["y"])
print("Explanation:", result.get("explanation"))

print("\n" + "=" * 60)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 60)


END-TO-END AI REQUIREMENTS CLASSIFICATION

[STAGE 1] DATA UNDERSTANDING
Original Input X: The application shall maintain 99.9% availability during each calendar month.
Normalized Input: The application shall maintain 99.9% availability during each calendar month.

[STAGE 2] REASONING
Requesting a brief explanation from Gemini...

[STAGE 3] INFERENCE
Sending requirement to Gemini API...

[STAGE 4] OUTPUT GENERATION
Identified clues: maintain 99.9% availability during each calendar month
Predicted output y: NFR
Explanation: The requirement specifies a system availability metric rather than a specific business function.

PIPELINE EXECUTION COMPLETE


### End-to-End Pipeline Results and Discussion

The proposed AI Requirements Classification Agent was demonstrated in Google Colab using the Gemini API. The agent receives natural-language software requirements as input X and processes them through data understanding, reasoning, inference, and output generation.

The demonstrated test requirement specifies that the application must maintain 99.9% availability during each calendar month. The expected classification is NFR because availability represents a reliability-related quality constraint.

The execution logs demonstrate how the input passes through the conceptual processing stages and how Gemini generates a classification with an explanation. The final output can be compared against the expected label to evaluate agreement for this test case.

This demonstration establishes the operation of the Gemini-based classification workflow for a selected observation. Further experiments using a larger labelled dataset are necessary to evaluate overall classification performance.